In [1]:
"""
reproduce_claims.py -- recompute every number quoted in the findings from the master CSV.

Usage:   python reproduce_claims.py master_robustness_evaluation_results.csv
Needs:   pandas, numpy, scipy
Note:    permutation tests are random (seed fixed below). Their p-values can move by ~0.01-0.02
         with a different seed; the Wilcoxon p-values and all other numbers are deterministic.

Vocabulary
  block   = one (dataset, model) pair, compared across training regimens  -> 12 blocks
  group   = one (dataset, perturbation) pair; contains the 12 runs (3 models x 4 regimens) -> 36 groups
  rauc    = retention AUC = area under (accuracy / clean accuracy) over intensity 0..1
  ret50   = accuracy at intensity 0.5 divided by clean accuracy (exact sampled point on every curve)
"""
import sys, ast
import numpy as np
import pandas as pd
from scipy import stats

PATH = "master_robustness_evaluation_results.csv"
rng = np.random.default_rng(0)
trap = getattr(np, "trapezoid", None) or np.trapz        # works on old and new numpy
N_PERM = 3000


def head(t):
    print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)


# ----------------------------------------------------------------------------------------------
# 0. LOAD + DERIVED METRICS
# ----------------------------------------------------------------------------------------------
raw = pd.read_csv(PATH)
df = raw.rename(columns={
    "Dataset": "ds", "Training_Regimen": "reg", "Model": "model", "Perturbation": "pert",
    "Clean_Test_Acc (%)": "test", "Clean_MCC": "cmcc", "Clean_Macro_F1": "cf1",
    "Critical_Threshold": "thr", "Stress_Accuracy (%)": "sacc", "Stress_MCC": "mcc",
    "Stress_Macro_F1": "f1", "Flip_Rate (%)": "flip",
    "Clean_ECE": "cece", "Clean_NLL": "cnll", "Stress_ECE": "ece", "Stress_NLL": "nll"})
df["cx"] = df.Curve_Intensities.apply(ast.literal_eval)
df["cy"] = df.Curve_Accuracies.apply(ast.literal_eval)
df["y0"] = df.cy.apply(lambda y: y[0])                                   # accuracy at intensity 0 (on the curve subset)
df["rauc"] = [trap(np.array(y) / y0, x) for x, y, y0 in zip(df.cx, df.cy, df.y0)]
df["ret50"] = [np.interp(0.5, x, y) / y0 for x, y, y0 in zip(df.cx, df.cy, df.y0)]
df["range"] = df.cy.apply(lambda y: max(y) - min(y))
print(f"Loaded {len(df)} rows: {df.ds.nunique()} datasets x {df.model.nunique()} models x "
      f"{df.reg.nunique()} regimens x {df.pert.nunique()} perturbations")

GROUPS = {"brightness": "in-recipe (bright/contrast/rot)", "contrast": "in-recipe (bright/contrast/rot)",
          "rotation": "in-recipe (bright/contrast/rot)",
          "gaussian_blur": "not in recipe (blur/noise/sensor)", "gaussian_noise": "not in recipe (blur/noise/sensor)",
          "sensor_interference": "not in recipe (blur/noise/sensor)",
          "occlusion": "occlusion", "environmental_obstruction": "compound (env/capture)",
          "capture_instability": "compound (env/capture)"}

# ----------------------------------------------------------------------------------------------
# 1. CLAIM 1 / 3: regimen effect on robustness (paired, 12 blocks, Wilcoxon signed-rank)
# ----------------------------------------------------------------------------------------------
head("1. Regimen effect on retention AUC (regimen minus baseline, paired over 12 blocks)")
base = df[df.reg == "baseline"].set_index(["ds", "model", "pert"]).rauc
for reg in ["augmented", "augmented_tuned", "tuned"]:
    x = df[df.reg == reg].set_index(["ds", "model", "pert"]).rauc
    d = (x - base).rename("d").reset_index()
    d["grp"] = d.pert.map(GROUPS)
    print(f"\n--- {reg} ---   (mean change | blocks improved | Wilcoxon p)")
    for p, g in d.groupby("pert"):
        print(f"  {p:27s} {g.d.mean():+.3f} | {(g.d > 0).sum():2d}/12 | p={stats.wilcoxon(g.d).pvalue:.4f}")
    print("  grouped (average the perturbations in the group within each block first, then test n=12):")
    for grp, g in d.groupby("grp"):
        m = g.groupby(["ds", "model"]).d.mean()
        print(f"  {grp:36s} {m.mean():+.3f} | {(m > 0).sum():2d}/12 | p={stats.wilcoxon(m).pvalue:.4f}")

# ----------------------------------------------------------------------------------------------
# 2. CLAIM 2: cliff location in physical units
# ----------------------------------------------------------------------------------------------
head("2. Median critical threshold, baseline vs augmented (physical units)")
PHYS = {"rotation": (lambda i: 90 * i, "degrees"), "brightness": (lambda i: 0.6 * i, "additive offset"),
        "contrast": (lambda i: 1 + 2 * i, "contrast factor"), "gaussian_noise": (lambda i: 0.35 * i, "noise sigma"),
        "gaussian_blur": (lambda i: 0.1 + 4.9 * i, "blur sigma (px)")}
for p, (f, unit) in PHYS.items():
    b = df[(df.pert == p) & (df.reg == "baseline")].thr.median()
    a = df[(df.pert == p) & (df.reg == "augmented")].thr.median()
    print(f"  {p:16s} {unit:16s} {f(b):6.2f} -> {f(a):6.2f}")
for p in ["occlusion", "environmental_obstruction", "capture_instability", "sensor_interference"]:
    b = df[(df.pert == p) & (df.reg == "baseline")].thr.median()
    a = df[(df.pert == p) & (df.reg == "augmented")].thr.median()
    print(f"  {p:27s} intensity {b:.2f} -> {a:.2f}")

# ----------------------------------------------------------------------------------------------
# 3. CLAIM 4: is clean accuracy a good guide to robustness?
# ----------------------------------------------------------------------------------------------
head("3. Clean accuracy vs robustness: correlation, regret, rank")
G = [g.reset_index(drop=True) for _, g in df.groupby(["ds", "pert"])]          # 36 groups of 12 runs
print(f"number of groups: {len(G)}, runs per group: {sorted(set(len(g) for g in G))}")

# 3a. mean Spearman between clean accuracy and rauc inside each group, with a shuffle null
obs_rho = np.mean([stats.spearmanr(g.test, g.rauc)[0] for g in G])
null_rho = [np.mean([stats.spearmanr(g.test, rng.permutation(g.rauc.values))[0] for g in G]) for _ in range(2000)]
print(f"\n3a. mean Spearman(clean test acc, retention AUC) = {obs_rho:.2f} ; "
      f"shuffle-null 95% band = [{np.percentile(null_rho, 2.5):.2f}, {np.percentile(null_rho, 97.5):.2f}]")
print("    by dataset:", {d: round(float(np.mean([stats.spearmanr(g.test, g.rauc)[0] for _, g in s.groupby('pert')])), 2)
                          for d, s in df.groupby("ds")})

# 3b. REGRET.  Per group: pick the run with the best clean accuracy; regret = best rauc in group - picked rauc.
regret_pick, regret_rand, ranks = [], [], []
for g in G:
    pick = g.loc[g.test.idxmax()]                       # run chosen by clean accuracy
    regret_pick.append(g.rauc.max() - pick.rauc)        # what we lose vs the best run
    regret_rand.append(g.rauc.max() - g.rauc.mean())    # expected loss of a uniformly random pick
    ranks.append((g.rauc > pick.rauc).sum() + 1)        # rank of the pick (1 = best)
obs = np.mean(regret_pick)
print(f"\n3b. regret of picking by clean accuracy = {obs:.3f} ; random-pick regret = {np.mean(regret_rand):.3f} ; "
      f"mean rank of pick = {np.mean(ranks):.1f}/12 ; picked the best run in {sum(r == 1 for r in ranks)}/{len(G)} groups")

# 3c. permutation test for the regret: simulate random picks
null = [np.mean([g.rauc.max() - g.rauc.iloc[rng.integers(len(g))] for g in G]) for _ in range(N_PERM)]
print(f"3c. p(random picking has regret <= observed) = {np.mean(np.array(null) <= obs):.3f}  "
      f"(large p = no evidence that clean accuracy beats random)")

# 3d. regimen choice inside each block: clean accuracy vs mean robustness
agree, rows = 0, []
for (d, m), g in df.groupby(["ds", "model"]):
    clean = g.groupby("reg").test.first()
    robust = g.groupby("reg").rauc.mean()                # mean over the 9 perturbations
    agree += clean.idxmax() == robust.idxmax()
    rows.append((d, m, clean["augmented"] - clean["baseline"], robust["augmented"] - robust["baseline"]))
print(f"\n3d. regimen picked by clean accuracy == regimen picked by robustness in {agree}/12 blocks (chance = 3/12)")
t = pd.DataFrame(rows, columns=["dataset", "model", "d_clean_acc(aug-base)", "d_rauc(aug-base)"]).round(3)
print(t[t.model == "Baseline_CNN"].to_string(index=False))

# ----------------------------------------------------------------------------------------------
# 4. CLAIMS 5-8: noise blind spot, ResNet-18, architecture profiles, best variants
# ----------------------------------------------------------------------------------------------
head("4. Noise, architectures, best variants")
n = df[df.pert == "gaussian_noise"]
print("median noise cliff sigma by regimen:", (0.35 * n.groupby("reg").thr.median()).round(3).to_dict())
print("median noise cliff sigma by model  :", (0.35 * n.groupby("model").thr.median()).round(3).to_dict())
pv = n.pivot_table(index=["ds", "reg"], columns="model", values="rauc")           # 16 contexts
other = pv[["Baseline_CNN", "MobileNetV2"]].max(axis=1)
print(f"ResNet18 has best noise rauc in {(pv.idxmax(axis=1) == 'ResNet18').sum()}/16 contexts, worst in "
      f"{(pv.idxmin(axis=1) == 'ResNet18').sum()}/16 ; Wilcoxon p (ResNet18 - best other) = "
      f"{stats.wilcoxon(pv.ResNet18 - other).pvalue:.4f}")

rows = [(p, g.loc[g.rauc.idxmax(), "model"], g.loc[g.rauc.idxmin(), "model"])
        for (_, _, p), g in df.groupby(["ds", "reg", "pert"])]
w = pd.DataFrame(rows, columns=["pert", "best", "worst"])
print("\nbest model per (dataset, regimen, perturbation) context, counts of 16:")
print(pd.crosstab(w.pert, w.best).to_string())
print("worst-model counts:")
print(pd.crosstab(w.pert, w.worst).to_string())

print("\nmean retention AUC by model   :", df.groupby("model").rauc.mean().round(3).to_dict())
print("mean retention AUC by regimen :", df.groupby("reg").rauc.mean().round(3).to_dict())
rv = df.groupby(["ds", "reg", "model"]).agg(rauc=("rauc", "mean"), clean=("test", "first")).reset_index()
print("\nbest variant per dataset (by mean retention AUC) vs best by clean accuracy:")
for d, g in rv.groupby("ds"):
    a, c = g.sort_values("rauc", ascending=False).iloc[0], g.sort_values("clean", ascending=False).iloc[0]
    print(f"  {d:13s} robust: {a.reg}/{a.model} rauc={a.rauc:.2f} clean={a.clean:.1f}   |   "
          f"clean-best: {c.reg}/{c.model} rauc={c.rauc:.2f} clean={c.clean:.1f}")

# ----------------------------------------------------------------------------------------------
# 5. CLAIM 9: MCC / macro-F1 vs accuracy, balanced vs imbalanced data
# ----------------------------------------------------------------------------------------------
head("5. MCC / macro-F1 vs accuracy")
df["r_acc"], df["r_mcc"], df["r_f1"] = df.sacc / df.test, df.mcc / df.cmcc, df.f1 / df.cf1
print("R^2 of stress MCC / stress macro-F1 regressed on stress accuracy, within each dataset:")
for d, g in df.groupby("ds"):
    print(f"  {d:13s} MCC {stats.linregress(g.sacc, g.mcc).rvalue ** 2:.2f}   F1 {stats.linregress(g.sacc, g.f1).rvalue ** 2:.2f}")
print("\nmean clean accuracy / macro-F1 / MCC by dataset:")
print(df.groupby("ds")[["test", "cf1", "cmcc"]].mean().round(3).to_string())
print("\nretention at the stress point (stress metric / clean metric), same intensity within each row:")
print(df.groupby("ds")[["r_acc", "r_mcc", "r_f1"]].mean().round(3).to_string())
for lab, s in [("ISIC", df[df.ds == "isic"]), ("other datasets", df[df.ds != "isic"])]:
    print(f"share of rows where MCC retention is >0.15 below accuracy retention, {lab}: "
          f"{100 * ((s.r_acc - s.r_mcc) > 0.15).mean():.0f}%")

# ----------------------------------------------------------------------------------------------
# 6. SANITY / CAVEAT NUMBERS
# ----------------------------------------------------------------------------------------------
head("6. Sanity checks and caveats")
print(f"mean stress accuracy at threshold {df.sacc.mean():.1f}% vs mean clean test accuracy {df.test.mean():.1f}%")
print(f"mean flip rate at threshold {df.flip.mean():.1f}%  (so ~{100 - df.flip.mean():.0f}% of predictions unchanged)")
ni = df[df.ds != "isic"]
print(f"non-ISIC rows with stress accuracy < 25% (chance = 10%): {(ni.sacc < 25).sum()}/{len(ni)}")
print("share of nearly flat curves (full range < 10 points), by dataset:",
      df.assign(flat=df.range < 10).groupby("ds").flat.mean().round(2).to_dict())
print("every curve has a sampled point at 0.5 and 1.0 and a largest gap of:",
      sorted(set(float(round(max(np.diff(x)), 2)) for x in df.cx)))


# ----------------------------------------------------------------------------------------------
# 7. FINAL CHECKS (robustness of the claims)
# ----------------------------------------------------------------------------------------------
head("7. Final checks")


def shift(reg, col, drop_ds=None, ref="baseline"):
    b = df[df.reg == ref].set_index(["ds", "model", "pert"])[col]
    x = df[df.reg == reg].set_index(["ds", "model", "pert"])[col]
    d = (x - b).rename("d").reset_index()
    d["grp"] = d.pert.map(GROUPS)
    return d if drop_ds is None else d[d.ds != drop_ds]


print("7a. augmented - baseline, in-recipe vs not-in-recipe vs occlusion; with and without ISIC; two measures")
for col in ["rauc", "ret50"]:
    for ex in [None, "isic"]:
        d, out = shift("augmented", col, ex), []
        for grp in ["in-recipe (bright/contrast/rot)", "not in recipe (blur/noise/sensor)", "occlusion"]:
            m = d[d.grp == grp].groupby(["ds", "model"]).d.mean()
            out.append(f"{grp.split(' (')[0]} {m.mean():+.3f} ({(m > 0).sum()}/{len(m)}, p={stats.wilcoxon(m).pvalue:.4f})")
        print(f"  {col:6s} {'all datasets' if ex is None else 'excl. ISIC ':12s} " + " | ".join(out))

print("\n7b. bootstrap 95% CI for the mean change on blur/noise/sensor (how big could a hidden effect be?)")
for reg in ["augmented", "augmented_tuned"]:
    d = shift(reg, "rauc")
    m = d[d.grp == "not in recipe (blur/noise/sensor)"].groupby(["ds", "model"]).d.mean().values
    bs = [rng.choice(m, len(m)).mean() for _ in range(10000)]
    print(f"  {reg:16s} mean {m.mean():+.3f}  95% CI [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")

print("\n7c. noise-floor proxy: tuned-vs-baseline differences should be ~null, so their spread bounds run-to-run noise")
t = shift("tuned", "rauc")
thr95 = np.percentile(t.d.abs(), 95)
print(f"  tuned - baseline (cell level): SD {t.d.std():.3f}, 95% of |diff| below {thr95:.3f}")
a = shift("augmented", "rauc")
for grp in ["in-recipe (bright/contrast/rot)", "occlusion"]:
    x = a[a.grp == grp].d
    print(f"  augmented - baseline, {grp.split(' (')[0]:10s}: mean {x.mean():+.3f} = {x.mean() / t.d.std():.1f} x that SD ; "
          f"{100 * (x > thr95).mean():.0f}% of individual cells exceed the 95% noise bound")

print("\n7d. ONE pick per dataset (12 variants): clean-best vs most robust (mean retention AUC over 9 perturbations)")
rv = df.groupby(["ds", "reg", "model"]).agg(rauc=("rauc", "mean"), clean=("test", "first")).reset_index()
for d_, g in rv.groupby("ds"):
    pk = g.loc[g.clean.idxmax()]
    print(f"  {d_:13s} clean-best = {pk.reg}/{pk.model}: regret {g.rauc.max() - pk.rauc:.3f} (random {g.rauc.max() - g.rauc.mean():.3f}), "
          f"robustness rank {(g.rauc > pk.rauc).sum() + 1}/12")

print("\n7e. regret of the clean-best pick under other targets / without ISIC (36 or 27 groups; permutation p)")
for lab, target, ex in [("rauc, all", "rauc", None), ("rauc, excl ISIC", "rauc", "isic"),
                        ("ret50, all", "ret50", None), ("threshold, all", "thr", None)]:
    GG = [g for g in G if ex is None or g.ds[0] != ex]
    o = np.mean([g[target].max() - g.loc[g.test.idxmax(), target] for g in GG])
    r = np.mean([g[target].max() - g[target].mean() for g in GG])
    nl = [np.mean([g[target].max() - g[target].iloc[rng.integers(len(g))] for g in GG]) for _ in range(N_PERM)]
    print(f"  {lab:16s} n={len(GG)}: clean-best {o:.3f} vs random {r:.3f}, p={np.mean(np.array(nl) <= o):.2f}")

print("\n7f. best variant (augmented ResNet18): retention at intensity 0.5, mean over datasets")
b = df[(df.reg == "augmented") & (df.model == "ResNet18")]
print("  ", b.groupby("pert").ret50.mean().round(2).sort_values().to_dict())
print("   ResNet18 noise retention@0.5 by regimen:",
      df[(df.model == "ResNet18") & (df.pert == "gaussian_noise")].groupby("reg").ret50.mean().round(2).to_dict())

print("\n7g. effect of augmentation on CLEAN test accuracy (augmented - baseline), points")
v = df.groupby(["ds", "model", "reg"]).test.first().unstack("reg")
print((v["augmented"] - v["baseline"]).unstack("model").round(1).to_string())

print(f"\n7h. calibration: clean ECE {df.cece.mean():.3f} -> ECE at threshold {df.ece.mean():.3f} ; "
      f"clean NLL {df.cnll.mean():.2f} -> {df.nll.mean():.2f} ; rows with ECE > 2x clean ECE: {100 * (df.ece > 2 * df.cece).mean():.0f}%")
print("   clean ECE by regimen:", df.groupby("reg").cece.mean().round(3).to_dict())

print("\n7i. agreement between cliff location and retention AUC (Spearman within perturbation, 48 runs each)")
print("  ", {p_: round(float(stats.spearmanr(g.thr, g.rauc)[0]), 2) for p_, g in df.groupby("pert")})


# ----------------------------------------------------------------------------------------------
# 8. COMPOUND PERTURBATIONS, SEVERITY CALIBRATION, MULTIPLE-COMPARISON CORRECTION
# ----------------------------------------------------------------------------------------------
head("8. Compounds, severity calibration, Holm correction")
COMP = {"sensor_interference": (["gaussian_blur", "gaussian_noise"], 0.8),
        "environmental_obstruction": (["gaussian_blur", "brightness"], 0.7),
        "capture_instability": (["rotation", "gaussian_blur"], 0.8)}
W = df.pivot_table(index=["ds", "reg", "model"], columns="pert", values="rauc")
Z = W.groupby(level="ds").transform(lambda s: (s - s.mean()) / s.std())      # standardise within dataset
print("8a. does a compound carry information beyond its two components? (retention AUC, n=48 runs)")
for c, (cs, s_) in COMP.items():
    r = [stats.spearmanr(Z[c], Z[k])[0] for k in cs]
    X = np.c_[Z[cs[0]], Z[cs[1]], np.ones(len(Z))]
    beta = np.linalg.lstsq(X, Z[c].values, rcond=None)[0]
    r2 = 1 - ((Z[c].values - X @ beta) ** 2).sum() / ((Z[c].values - Z[c].mean()) ** 2).sum()
    print(f"  {c:27s} rho with {cs[0]}: {r[0]:.2f} | with {cs[1]}: {r[1]:.2f} | R^2 on both = {r2:.2f}")

print("\n8b. which component does the compound's critical threshold track? (threshold of component / scale)")
T = df.pivot_table(index=["ds", "reg", "model"], columns="pert", values="thr")
for c, (cs, s_) in COMP.items():
    gaps = pd.concat([(T[c] - T[k] / s_).abs() for k in cs], axis=1, keys=cs)
    print(f"  {c:27s} closest component counts (of 48): {gaps.idxmin(axis=1).value_counts().to_dict()} ; "
          f"median thresholds: compound {T[c].median():.2f}, " + ", ".join(f"{k} {T[k].median():.2f}" for k in cs))

print("\n8c. equal-effect intensities: where the MEDIAN retention over all 48 runs falls to 0.8 / 0.5")
grid = np.linspace(0, 1, 201)
for p_, g in df.groupby("pert"):
    med = np.median([np.interp(grid, r.cx, np.array(r.cy) / r.y0) for r in g.itertuples()], axis=0)
    cross = lambda t: (round(float(grid[np.where(med <= t)[0][0]]), 2) if (med <= t).any() else "never")
    print(f"  {p_:27s} 0.8 at I={cross(0.8)} | 0.5 at I={cross(0.5)} | median retention at I=1: {med[-1]:.2f}")

print("\n8d. Holm correction over the 27 per-perturbation strategy tests (3 regimens x 9 perturbations)")
base = df[df.reg == "baseline"].set_index(["ds", "model", "pert"]).rauc
tests = []
for reg in ["augmented", "augmented_tuned", "tuned"]:
    x = df[df.reg == reg].set_index(["ds", "model", "pert"]).rauc
    d = (x - base).rename("d").reset_index()
    for p_, g in d.groupby("pert"):
        tests.append((reg, p_, g.d.mean(), int((g.d > 0).sum()), stats.wilcoxon(g.d).pvalue))
T2 = pd.DataFrame(tests, columns=["regimen", "pert", "mean", "blocks_up", "p"]).sort_values("p").reset_index(drop=True)
m_ = len(T2)
T2["holm"] = np.maximum.accumulate(np.minimum(1, (m_ - np.arange(m_)) * T2.p.values))
print(T2[T2.p < 0.05].round(4).to_string(index=False))
print(f"  raw p<0.05: {(T2.p < 0.05).sum()} tests ; still significant after Holm: {(T2.holm < 0.05).sum()}")

Loaded 432 rows: 4 datasets x 3 models x 4 regimens x 9 perturbations

1. Regimen effect on retention AUC (regimen minus baseline, paired over 12 blocks)

--- augmented ---   (mean change | blocks improved | Wilcoxon p)
  brightness                  +0.134 | 12/12 | p=0.0005
  capture_instability         +0.043 |  7/12 | p=0.0923
  contrast                    +0.122 | 12/12 | p=0.0005
  environmental_obstruction   +0.092 | 12/12 | p=0.0005
  gaussian_blur               -0.008 |  3/12 | p=0.1763
  gaussian_noise              +0.007 |  4/12 | p=0.5693
  occlusion                   +0.264 | 12/12 | p=0.0005
  rotation                    +0.101 | 11/12 | p=0.0010
  sensor_interference         -0.004 |  4/12 | p=0.5693
  grouped (average the perturbations in the group within each block first, then test n=12):
  compound (env/capture)               +0.067 | 12/12 | p=0.0005
  in-recipe (bright/contrast/rot)      +0.119 | 12/12 | p=0.0005
  not in recipe (blur/noise/sensor)    -0.002 |  5/12 